# 81 · Titanic — EDA y baseline

Segunda parte de la demostración. Aquí exploras los datos y entrenas un primer modelo *baseline*.
Lo registras **manualmente** en MLflow para que veas con detalle qué se guarda en un *run* y por qué.

### Objetivos de aprendizaje
- Explorar la relación entre variables (sexo, clase, edad) y la supervivencia.
- Entender la **tasa base** como punto de comparación honesto antes de modelar.
- Construir un `Pipeline` de SparkML mínimo y evaluarlo sobre un conjunto de validación.
- Registrar un *run* de MLflow a mano: parámetros, métricas, artefactos y el modelo con firma.

## 1. Parámetros y configuración de MLflow

Fijamos el registro de modelos en **Unity Catalog** (`databricks-uc`) y creamos el experimento bajo tu
carpeta personal `/Users/<tu_usuario>/titanic_mlflow`. En serverless el experimento nunca debe ir a
`/Shared`.

In [ ]:
import mlflow

CATALOGO = "big_data_ii_2025"
ESQUEMA = "spark_examples"
TABLA_BRONZE_TRAIN = f"{CATALOGO}.{ESQUEMA}.titanic_bronze_train"

mlflow.set_registry_uri("databricks-uc")

usuario = spark.sql("SELECT current_user()").first()[0]
RUTA_EXPERIMENTO = f"/Users/{usuario}/titanic_mlflow"
mlflow.set_experiment(RUTA_EXPERIMENTO)

print("Usuario     :", usuario)
print("Experimento :", RUTA_EXPERIMENTO)

## 2. Exploración de datos (EDA)

Cargamos la tabla bronze de entrenamiento y miramos cómo se relacionan algunas variables con la
supervivencia. La pregunta es: *¿qué personas tenían más probabilidad de sobrevivir?*

In [ ]:
from pyspark.sql import functions as F

df = spark.table(TABLA_BRONZE_TRAIN)
print("Filas:", df.count())

In [ ]:
# Tasa base: proporción de la clase mayoritaria (Survived = 0).
tasa_supervivencia = df.agg(F.avg("Survived")).first()[0]
tasa_base = 1 - tasa_supervivencia
print(f"Tasa de supervivencia: {tasa_supervivencia:.3f}")
print(f"Tasa base (predecir 'nadie sobrevive'): {tasa_base:.3f}")

# Verificación de la predicción: la tasa base ronda 0.616.
assert 0.60 <= tasa_base <= 0.63, f"Tasa base inesperada: {tasa_base}"
print("La tasa base ~0.616 es el umbral a superar.")

### Supervivencia por sexo y por clase

In [ ]:
print("Supervivencia por sexo:")
df.groupBy("Sex").agg(
    F.count("*").alias("n"),
    F.round(F.avg("Survived"), 3).alias("tasa_supervivencia"),
).orderBy("Sex").display()

In [ ]:
print("Supervivencia por clase:")
df.groupBy("Pclass").agg(
    F.count("*").alias("n"),
    F.round(F.avg("Survived"), 3).alias("tasa_supervivencia"),
).orderBy("Pclass").display()

## 3. Preparación del baseline

El baseline usa un conjunto **mínimo** de variables: `Pclass`, `Sex`, `Age`, `Fare`, `SibSp`, `Parch`.
Imputamos `Age` y `Fare` con una estrategia simple (mediana global vía `Imputer`).

### 3.1 División train/validación

Partimos `train.csv` en 80% entrenamiento y 20% validación con semilla fija. 

In [ ]:
df_ent, df_val = df.randomSplit([0.8, 0.2], seed=42)
print(f"Entrenamiento: {df_ent.count()} filas")
print(f"Validación:    {df_val.count()} filas")

### 3.2 Pipeline de SparkML

Encadenamos: imputación de numéricas → indexado de categóricas → *one-hot* → ensamblado de vector →
regresión logística. Todo dentro de un único `Pipeline` para que sea reproducible y serializable.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

FEATURES_E1 = ["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch"]

# Imputación de numéricas con la mediana (estrategia simple del baseline).
imputer = Imputer(
    inputCols=["Age", "Fare"],
    outputCols=["Age_imp", "Fare_imp"],
    strategy="median",
)

# Sexo: índice + one-hot.
sex_idx = StringIndexer(inputCol="Sex", outputCol="Sex_idx", handleInvalid="keep")
sex_ohe = OneHotEncoder(inputCols=["Sex_idx"], outputCols=["Sex_ohe"])

assembler = VectorAssembler(
    inputCols=["Pclass", "Sex_ohe", "Age_imp", "Fare_imp", "SibSp", "Parch"],
    outputCol="features",
    handleInvalid="keep",
)

lr = LogisticRegression(featuresCol="features", labelCol="Survived", maxIter=50)

pipeline_e1 = Pipeline(stages=[imputer, sex_idx, sex_ohe, assembler, lr])

## 4. Entrenamiento y registro manual en MLflow

En **E1 registramos todo a mano** a propósito. Verás exactamente qué es un parámetro, qué es una
métrica, qué es un artefacto y cómo se loggea el modelo con su **firma**.

In [ ]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
)
from mlflow.models import infer_signature
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def evaluar(pred_df):
    """Devuelve accuracy, f1 y AUC-ROC sobre un DataFrame con columnas prediction/probability/Survived."""
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="accuracy")
    f1_eval = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="f1")
    auc_eval = BinaryClassificationEvaluator(
        labelCol="Survived", rawPredictionCol="probability", metricName="areaUnderROC")
    return (acc_eval.evaluate(pred_df),
            f1_eval.evaluate(pred_df),
            auc_eval.evaluate(pred_df))


def graficar_matriz_confusion(pred_df, ruta_png, titulo):
    """Genera una matriz de confusión 2x2 como PNG desde predicciones de Spark."""
    conteos = (pred_df.groupBy("Survived", "prediction").count()
               .toPandas())
    matriz = [[0, 0], [0, 0]]
    for _, fila in conteos.iterrows():
        matriz[int(fila["Survived"])][int(fila["prediction"])] = int(fila["count"])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(matriz, cmap="Oranges")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, matriz[i][j], ha="center", va="center", fontsize=14)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["No", "Sí"]); ax.set_yticklabels(["No", "Sí"])
    ax.set_title(titulo)
    fig.tight_layout()
    fig.savefig(ruta_png, dpi=100)
    plt.close(fig)

In [ ]:
with mlflow.start_run(run_name="E1_baseline_logreg") as run:
    # Etiquetas descriptivas del run.
    mlflow.set_tags({
        "experimento": "E1",
        "modelo": "LogisticRegression",
        "notebook": "81",
    })

    # Entrenamiento.
    modelo_e1 = pipeline_e1.fit(df_ent)

    # Predicciones sobre entrenamiento y validación.
    pred_ent = modelo_e1.transform(df_ent)
    pred_val = modelo_e1.transform(df_val)

    acc_ent, _, _ = evaluar(pred_ent)
    acc_val, f1_val, auc_val = evaluar(pred_val)

    # Parámetros (lo que configuramos nosotros).
    mlflow.log_param("features", ", ".join(FEATURES_E1))
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("seed", 42)

    # Métricas (lo que mide el modelo).
    mlflow.log_metric("accuracy_train", acc_ent)
    mlflow.log_metric("accuracy", acc_val)      # accuracy de validación = métrica principal
    mlflow.log_metric("f1", f1_val)
    mlflow.log_metric("auc_roc", auc_val)

    # Artefacto: matriz de confusión de validación.
    ruta_cm = "/tmp/cm_e1.png"
    graficar_matriz_confusion(pred_val, ruta_cm, "E1 · Matriz de confusión (validación)")
    mlflow.log_artifact(ruta_cm, artifact_path="graficos")

    # Modelo con firma e input_example (requisito para registrar en Unity Catalog más adelante).
    ejemplo = df_ent.select(*FEATURES_E1, "Survived").limit(5).toPandas()
    firma = infer_signature(
        df_ent.select(*FEATURES_E1).limit(5).toPandas(),
        pred_val.select("prediction").limit(5).toPandas(),
    )
    mlflow.spark.log_model(
        modelo_e1,
        artifact_path="model",
        signature=firma,
        input_example=ejemplo.drop(columns=["Survived"]),
    )

    run_id_e1 = run.info.run_id

print("--- Resultados E1 (baseline) ---")
print(f"Accuracy entrenamiento : {acc_ent:.4f}")
print(f"Accuracy validación    : {acc_val:.4f}")
print(f"F1 validación          : {f1_val:.4f}")
print(f"AUC-ROC validación     : {auc_val:.4f}")
print(f"run_id                 : {run_id_e1}")

## 5. Interpreta los resultados

- El *accuracy* de validación del baseline suele rondar **0.78–0.80**, claramente por encima de la
  tasa base (~0.616). Ya con seis variables el modelo aporta valor.
- Normalmente el *accuracy* de entrenamiento es algo mayor que el de validación: el modelo se ajusta
  un poco a los datos que vio. Confirma si eso ocurrió en tu ejecución.

## 6. Explora el run en la interfaz de Databricks

Antes de seguir, podemos revisar el *run* anterior:
1. En la barra lateral, dentro de *Machine Learning*, abrir **Experiments**.
2. Abre el experimento `titanic_mlflow` y la tabla de *runs*.
3. Abre el *run* **`E1_baseline_logreg`** y en los tabs:
   - **Overview:** parámetros (`features`, `maxIter`, `seed`) y métricas (`accuracy`, `f1`, `auc_roc`).
   - **Artifacts:** la carpeta `model` (el pipeline serializado) y `graficos/cm_e1.png` (la matriz de confusión).

En esta vista es posible ver los *runs* y compararlos